# Day 04 拓展练习：淘宝商品数据清洗与特征构造

**数据文件：** 淘宝全品类全国数据.csv

**适用对象：** 已完成上午主练习的同学。

## 学习目标

- 按业务含义处理服务字段与商品属性字段的缺失值；
- 清理商品 ID 中的隐藏空白字符；
- 将销量区间文本转换为可分析的数值特征；
- 构造价格分层字段；
- 导出清洗后的商品数据。

---
## 1. 读取数据

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/淘宝全品类全国数据.csv"),
    Path("data/淘宝全品类全国数据.csv"),
    Path("/Users/yq/muc_training/data/淘宝全品类全国数据.csv"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到淘宝全品类全国数据.csv，请修改 DATA_PATH。")

taobao = pd.read_csv(DATA_PATH)
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{taobao.shape[0]} 行，{taobao.shape[1]} 列")
taobao.head()

读取文件：..\data\淘宝全品类全国数据.csv
数据形状：25000 行，15 列


,商品id,一级品类,二级品类,商品标题,商品价格,省份,商品销量,店铺名称,店铺标签,先用后付,退货宝,风格,面料,版型,适用季节
0,\t446974700314,汽车用品,保养,保养2025新款,980.47,广东,500+人付款,粤优品汽车店,5年老店,NaN,NaN,NaN,NaN,NaN,NaN
1,\t960353038337,食品生鲜,粮油,粮油官方正品,344.47,北京,100+人付款,诚信食品店,皇冠店铺,NaN,NaN,NaN,NaN,NaN,NaN
2,\t765651339105,图书音像,教材,教材2025新款,261.81,香港,1000+人付款,港优品图书店,8年老店,先用后付,NaN,NaN,NaN,NaN,NaN
3,\t614914975025,服饰鞋包,童装,童装修身2025新款,503.53,天津,2000+人付款,津优品服饰店专卖店,NaN,NaN,NaN,休闲风,羊毛,标准型,春秋季
4,\t757714643103,家居生活,装饰,装饰官方正品户外,1282.75,北京,500+人付款,时尚家居店旗舰店,回头客3千,NaN,NaN,NaN,NaN,NaN,NaN


### 任务 1：数据质量体检

请完成以下检查：

1. 查看字段类型；
2. 创建缺失值报告；
3. 统计完全重复行；
4. 观察商品销量字段的取值。

In [2]:
# TODO：完成数据质量检查
taobao.info()
missing_report = pd.DataFrame({
    "缺失数量": taobao.isna().sum(),
    "缺失比例(%)": (taobao.isna().mean() * 100).round(2)
})
missing_report = missing_report[missing_report["缺失数量"] > 0].sort_values("缺失数量", ascending=False)
print(f"\nMissing:\n{missing_report.to_string()}")
print(f"\nDuplicate rows: {taobao.duplicated().sum()}")
print(f"\nSales values:\n{taobao['商品销量'].value_counts()}")

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   商品id    25000 non-null  str    
 1   一级品类    25000 non-null  str    
 2   二级品类    25000 non-null  str    
 3   商品标题    25000 non-null  str    
 4   商品价格    25000 non-null  float64
 5   省份      25000 non-null  str    
 6   商品销量    25000 non-null  str    
 7   店铺名称    25000 non-null  str    
 8   店铺标签    21259 non-null  str    
 9   先用后付    3158 non-null   str    
 10  退货宝     2724 non-null   str    
 11  风格      3988 non-null   str    
 12  面料      2375 non-null   str    
 13  版型      2036 non-null   str    
 14  适用季节    4822 non-null   str    
dtypes: float64(1), str(14)
memory usage: 2.9 MB

Missing:
       缺失数量  缺失比例(%)
版型    22964    91.86
面料    22625    90.50
退货宝   22276    89.10
先用后付  21842    87.37
风格    21012    84.05
适用季节  20178    80.71
店铺标签   3741    14.96

Duplicate rows: 0

Sales values:
商品销量
1000+人付

---
## 2. 清理商品 ID

商品 ID 是标识符，不应转换为数值。本数据的商品 ID 可能带有制表符或首尾空白字符，这会影响去重、匹配和导出。

### 任务 2：清理隐藏空白

先用 repr 查看第一个商品 ID 的原始形式，再使用字符串方法清理首尾空白。

In [4]:
# TODO：观察原始值
print(f"\nBefore: {repr(taobao.loc[0, '商品id'])}")
taobao["商品id"] = taobao["商品id"].astype(str).str.strip()
print(f"After: {repr(taobao.loc[0, '商品id'])}")



Before: '\\t446974700314'
After: '\\t446974700314'


---
## 3. 按业务语义处理缺失值

请区分两类字段：

- 服务字段：先用后付、退货宝。空值可解释为未提供服务；
- 商品属性字段：风格、面料、版型、适用季节。空值更适合解释为商家未标注。

注意：这是一项业务假设，应在分析报告中写清楚。

### 任务 3：完成缺失值填补

请将服务字段填为“未提供”，商品属性字段填为“未标注”，并检查处理后的缺失数量。

In [5]:
service_cols = ["先用后付", "退货宝"]
attribute_cols = ["风格", "面料", "版型", "适用季节"]

# TODO：填补服务字段与商品属性字段
service_cols = ["先用后付", "退货宝"]
attribute_cols = ["风格", "面料", "版型", "适用季节"]
for col in service_cols:
    taobao[col] = taobao[col].fillna("未提供")
for col in attribute_cols:
    taobao[col] = taobao[col].fillna("未标注")
print(f"\nAfter fill:\n{taobao[service_cols + attribute_cols].isna().sum()}")


After fill:
先用后付    0
退货宝     0
风格      0
面料      0
版型      0
适用季节    0
dtype: int64


---
## 4. 销量文本转换

商品销量是区间文本，例如“500+人付款”和“1万+人付款”。它不能直接参与平均值、排序或相关性计算。

本练习提取该区间的**最低销量门槛**，字段命名为“销量下限”。它不是实际销量。

### 任务 4：创建销量下限

请补全函数，将“1000+人付款”转换为 1000，将“1万+人付款”转换为 10000。

In [6]:

def sales_lower_bound(value):
    value = str(value).replace("+人付款", "")
    if "万" in value:
        return float(value.replace("万", "")) * 10000
    return float(value)

taobao["销量下限"] = taobao["商品销量"].apply(sales_lower_bound)
print(f"\nSales conversion:\n{taobao[['商品销量', '销量下限']].drop_duplicates().sort_values('销量下限').to_string()}")



Sales conversion:
         商品销量       销量下限
1     100+人付款     100.00
7     200+人付款     200.00
0     500+人付款     500.00
2    1000+人付款    1000.00
3    2000+人付款    2000.00
25   5000+人付款    5000.00
14     1万+人付款   10000.00
62     2万+人付款   20000.00
80     5万+人付款   50000.00
8     10万+人付款  100000.00
31    20万+人付款  200000.00
15    50万+人付款  500000.00
218  100万+人付款 1000000.00


---
## 5. 价格分层与候选异常值

价格为连续数值。先用 IQR 方法识别候选异常值；随后创建“价格区间”字段，便于后续按价格层级分析。

In [8]:
def iqr_outlier_summary(series):
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 5：检查价格并构造价格区间

1. 输出商品价格的候选异常值摘要；
2. 创建“价格区间”：0-100、100-500、500-1000、1000-3000、3000 以上；
3. 统计每个价格区间的商品数量。

提醒：高价商品可能是正常商品，不要仅因它被 IQR 识别出来就删除。

In [10]:
# TODO：检查商品价格的候选异常值
def iqr_outlier_summary(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return pd.Series({"Q1": q1, "Q3": q3, "下限": q1-1.5*iqr, "上限": q3+1.5*iqr, "候选异常值数量": ((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).sum()})
    
bins = [0, 100, 500, 1000, 3000, np.inf]
labels = ["0-100元", "100-500元", "500-1000元", "1000-3000元", "3000元以上"]
taobao["价格区间"] = pd.cut(taobao["商品价格"], bins=bins, labels=labels, right=True)
print(f"\nPrice tiers:\n{taobao['价格区间'].value_counts().sort_index()}")
print("Note: High-price items may be normal - don't delete based on IQR alone")



Price tiers:
价格区间
0-100元        2068
100-500元      8933
500-1000元     5925
1000-3000元    6794
3000元以上       1280
Name: count, dtype: int64
Note: High-price items may be normal - don't delete based on IQR alone


---
## 6. 验收与导出

提交前，确保已生成销量下限和价格区间，且指定服务、属性字段没有缺失。

In [11]:
assert taobao[service_cols + attribute_cols].isna().sum().sum() == 0
assert "销量下限" in taobao.columns
assert "价格区间" in taobao.columns

OUTPUT_PATH = Path("../output/taobao_product_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
taobao.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\nExported: {OUTPUT_PATH.resolve()}")
print(f"Rows: {len(taobao)}, Cols: {taobao.shape[1]}")



Exported: C:\Users\a1098\Desktop\firstgame\shixi\muc-commerce-3-24012453\output\taobao_product_cleaned.csv
Rows: 25000, Cols: 17


## 提交前自查

- [ ] 已完成缺失值报告与重复值检查；
- [ ] 商品 ID 已清理首尾空白；
- [ ] 服务字段填为“未提供”；
- [ ] 属性字段填为“未标注”；
- [ ] 已创建“销量下限”；
- [ ] 已创建“价格区间”；
- [ ] 已导出 taobao_product_cleaned.csv。

## 思考题

“销量下限”与真实销量有什么区别？如果要估算商品销售额，为什么不能直接用“商品价格 × 销量下限”作为真实销售额？